Industry-Style Agentic RAG with LangGraph
Why this is industry-style
A simple RAG system searches only the private knowledge base.

An industry-style Agentic RAG system should do this:

Question
  ↓
Search private knowledge base first
  ↓
Grade the private evidence
  ↓
If private evidence is good → answer from KB
  ↓
If private evidence is weak → search the web
  ↓
Grade web evidence
  ↓
Generate a grounded answer with source type
This design is useful because private documents may be incomplete or outdated

In [ ]:
import os 

try:
    from dotenv import load_dotenv
    load_dotenv()
except:
    pass
if not os.environ.get("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY is not set")

if not os.environ.get("PINECONE_API_KEY"):
    raise ValueError("PINECONE_API_KEY is not set")

if not os.environ.get("TAVILY_API_KEY"):
    raise ValueError("TAVILY_API_KEY is not set")

print("All environment variables are set")









In [11]:

import requests
from bs4 import BeautifulSoup
from langchain_core.documents import Document

SOURCE_URL = "https://docs.langchain.com/oss/python/langgraph/agentic-rag"

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; Agentic-RAG-Industry-Demo/1.0)"
}

response = requests.get(
    SOURCE_URL,
    headers=headers,
    timeout=30,
)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Remove elements that are not useful for RAG
for element in soup.select(
    "script, style, nav, footer, header, "
    "noscript, svg, form"
):
    element.decompose()

# Extract the main article when available
content = soup.select_one("main") or soup.body

if content is None:
    raise ValueError("No readable page content was found.")

page_text = content.get_text(separator="\n", strip=True)

if not page_text:
    raise ValueError("The page returned empty text.")

# Create a LangChain Document
raw_docs = [
    Document(
        page_content=page_text,
        metadata={"source": SOURCE_URL},
    )
]

print("Loaded documents:", len(raw_docs))
print("Source:", raw_docs[0].metadata["source"])
print("\nPreview:\n")
print(raw_docs[0].page_content[:1500])


Loaded documents: 1
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag

Preview:

Build a
retrieval
agent with LangGraph that decides when to search a vector store versus answering the user directly.
LangChain offers built-in
agent
implementations built on
LangGraph
primitives. When you need deeper customization, implement the agent directly in LangGraph. This tutorial walks through one retrieval-agent pattern.
In this tutorial you will:
Fetch and preprocess documents for retrieval.
Index those documents for semantic search and create a retriever tool for the agent.
Build an agentic RAG system that can decide when to use the retriever tool.
​
Concepts
This tutorial covers the following concepts:
Retrieval
using
document loaders
,
text splitters
,
embeddings
, and
vector stores
The LangGraph
Graph API
, including state, nodes, edges, and conditional edges.
​
Setup
Install the required packages and set your API keys:
pip install
-
U langgraph langchain langchain
-
openai

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True,)

chunks = splitter.split_documents(raw_docs)
print("Number of chunks created:", len(chunks))
print(
"Preview of first chunk:\n")
print(chunks[0].page_content[:900]
)

Number of chunks created: 20
Preview of first chunk:

Build a
retrieval
agent with LangGraph that decides when to search a vector store versus answering the user directly.
LangChain offers built-in
agent
implementations built on
LangGraph
primitives. When you need deeper customization, implement the agent directly in LangGraph. This tutorial walks through one retrieval-agent pattern.
In this tutorial you will:
Fetch and preprocess documents for retrieval.
Index those documents for semantic search and create a retriever tool for the agent.
Build an agentic RAG system that can decide when to use the retriever tool.
​
Concepts
This tutorial covers the following concepts:
Retrieval
using
document loaders
,
text splitters
,
embeddings
, and
vector stores
The LangGraph
Graph API
, including state, nodes, edges, and conditional edges.
​
Setup
Install the required packages and set your API keys:
pip install
-
U langgraph langchain langchain
-
opena


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

sample_vector = embeddings.embed_query("What is Agentic RAG?")
print("Embedding dimensions:", len(sample_vector))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12173.50it/s]


Embedding dimensions: 384


In [16]:
# sample_vector

In [20]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import time

pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"), environment="us-east1-gcp")
#create index if only it does not exist
INDEX_NAME = "agentic-rag-index"
NAME_SPACE = "agentic-rag-namespace"
# Create the index only if it does not already exist.
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,       # all-MiniLM-L6-v2 embedding dimension
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )
    # Wait until Pinecone reports the new index as ready.
while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)

print("Pinecone index ready:", INDEX_NAME)

# Upload the document chunks and create the LangChain vector store.
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAME_SPACE,
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAME_SPACE,
    }
)

print("Pinecone vector database and retriever are ready.")

Pinecone index ready: agentic-rag-index
Pinecone vector database and retriever are ready.


In [22]:
# Load Existing Index

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = "agentic-rag-index"
NAME_SPACE = "agentic-rag-namespace"

# Connect to Pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Load existing Pinecone index
index = pc.Index(INDEX_NAME)

# Connect existing index with LangChain
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=NAME_SPACE,
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAME_SPACE,
    }
)

print("Existing Pinecone index loaded:", INDEX_NAME)

Existing Pinecone index loaded: agentic-rag-index


In [23]:
test_question = "What happens if retrieved documents are not relevant in Agentic RAG?"

kb_docs = retriever.invoke(test_question)

for i, doc in enumerate(kb_docs, 1):
    print(f"\n--- KB RESULT {i} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:700])


--- KB RESULT 1 ---
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
()))
​
Run the agentic RAG
Test the complete graph by running it with a question:
def
run_agentic_rag
()
->
None
:
for
chunk
in
graph
.
stream
(
{
"messages"
:
[
{
"role"
:
"user"
,
"content"
:
"What does Lilian Weng say about types of reward hacking?"
,
}
]
},
stream_mode
=
"values"
,
):
last_message
=
chunk
[
"
messages
"
][
-
1
]
pretty_print
=
getattr
(
last_message
,
"pretty_print"
,
None
)
if
callable
(
pretty_print
):
pretty_print
()
​
See also
Retrieval
Graph API
Agents
Build a RAG agent
Build a semantic search engine
Connect these docs
to your agent of choice via MCP for real-time answers.
Edit this page on GitHub
or
file an issue
.
Was this page helpful?
Yes
No

--- KB RESULT 2 ---
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
=
1
)
def
_get_retriever
():
vectorstore
=
InMemoryVectorStore
.
from_documents
(
documents
=
doc_splits
,
embedding
=
OpenAIEmbeddings
(),
)


In [24]:
from langchain_groq import ChatGroq

# Replace this model if your Groq dashboard shows a different available model.
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
)

print(llm.invoke("Explain Agentic RAG in one sentence.").content)

Agentic RAG is a retrieval‑augmented generation framework in which an autonomous agent actively queries, selects, and incorporates relevant external documents on the fly to guide and improve its generated responses.


In [25]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(
    max_results=5,
    topic="general",
    include_answer=True,
    include_raw_content=False,
)

print("Tavily search tool ready.")

Tavily search tool ready.
